# SPHERE-Count (Method 3)

Automated counting of microspheres in brightfield images.

## How to run
1. Open this notebook in Google Colab (or Jupyter).
2. Click **Runtime → Run all**.
3. When prompted, upload one or more images (JPG, PNG, or TIF).

## Output
For each image:
- the bead count — report the **BEST** value
- the image with every bead marked (red = single bead, orange = clump)
- a quality-control histogram

A `counts.csv` summary and a `results.zip` of all outputs are produced automatically.


In [ ]:
# Imports.
# In Google Colab these are already installed. On your own machine, if any
# is missing:  pip install numpy scipy scikit-image opencv-python pillow matplotlib

import io, os, csv, zipfile
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import feature, measure, morphology

print("Imports ready.")


In [ ]:
# Counting pipeline. Run this cell to define the functions; no edits needed.

"""
SPHERE-Count: area/absorbance pipeline for counting black paramagnetic
polyethylene microspheres in brightfield images.

This is "Method 3" from:
    Bakleh M, Shuaibu AS, Al-Maslamani NA. SPHERE-Count: An Automated
    Image-Analysis Pipeline for Density-Independent Bead Enumeration.
    (submitted to SLAS Technology, 2026)

The pipeline is segmentation-free in the sense that it does not attempt to
draw a boundary around every individual bead. Instead it derives the count
from the total foreground signal divided by the signal of a single bead,
which is why it stays accurate as beads touch and clump.

Overview of the method
----------------------
1. Relative absorbance
   Each pixel is expressed as how much darker it is than the local
   background (a large-kernel median). Beads are dark on a light field, so
   this yields a near-flat background at ~0 and beads approaching ~1,
   independent of uneven illumination.

2. Automatic bead-diameter estimate
   From the isolated, well-formed objects in the image, the median object
   area gives a characteristic single-bead diameter in pixels. Every
   downstream scale parameter (blob sigma, minimum separation, minimum
   object size) is derived from this, so the pipeline adapts to image
   resolution / bead size without manual tuning.

3. Candidate detection (Laplacian-of-Gaussian blob detection)
   A LoG filter tuned to the estimated bead size highlights bead centres;
   robust (MAD-based) thresholding plus local-maxima detection gives one
   candidate peak per resolvable bead. This produces `n_peak`, a
   conservative lower bound (only beads that can be individually resolved).

4. Aggregate quantification (the reported count)
   The foreground mask's total area and total absorbance ("mass") are each
   divided by the corresponding single-bead value, giving two density-robust
   estimates `n_area` and `n_mass`. Their mean, `best`, is the number the
   study reports.

Returned counts
---------------
best    : reported count = round((n_area + n_mass) / 2)
n_area  : total foreground area / single-bead area
n_mass  : total foreground absorbance / single-bead absorbance
n_peak  : resolvable-bead count (lower bound)

All tunable constants are exposed as arguments of `count_image` with the
exact defaults used for the manuscript results; see DEFAULTS below.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import cv2
from scipy import ndimage as ndi
from skimage import feature, measure, morphology


# ---------------------------------------------------------------------------
# Default parameters (exactly as used for the manuscript; do not silently
# change these — they define the published Method 3 configuration).
# ---------------------------------------------------------------------------
DEFAULTS = dict(
    bg_kernel=31,     # median-blur kernel (px) for local-background estimate
    pix_cut=0.30,     # relative-absorbance threshold for the foreground mask
    peak_cut=0.40,    # min relative absorbance for a candidate to count as a bead
    log_snr=3.5,      # robust (MAD) SNR threshold for LoG peak detection
    solidity=0.88,    # isolated-bead shape gate (calibration objects)
    eccentricity=0.70,
)


@dataclass
class BeadResult:
    """Container for a single image's counting result and intermediates."""
    best: int
    n_peak: int
    n_area: float
    n_mass: float
    diameter: float
    single_area: float
    # intermediates kept for overlay / QC plotting
    rel: np.ndarray = field(repr=False, default=None)
    peaks: np.ndarray = field(repr=False, default=None)
    valid: list = field(repr=False, default=None)
    npk: np.ndarray = field(repr=False, default=None)
    cand_abs: np.ndarray = field(repr=False, default=None)
    img: np.ndarray = field(repr=False, default=None)

    def as_row(self) -> dict:
        """Flat dict of the numeric results, for CSV export."""
        return dict(
            beads_best=self.best,
            n_peak=self.n_peak,
            n_area=round(self.n_area),
            n_mass=round(self.n_mass),
            bead_diameter_px=round(self.diameter, 2),
        )


def relative_absorbance(img8: np.ndarray, bg_kernel: int = 31) -> np.ndarray:
    """Per-pixel darkness relative to the local background.

    Parameters
    ----------
    img8 : 2-D uint8 array
        Grayscale image (beads dark on a light field).
    bg_kernel : int
        Median-blur kernel size in pixels; forced odd. Should be several
        times the bead diameter so single beads do not bias their own
        background.

    Returns
    -------
    2-D float32 array in roughly [0, 1]: ~0 background, higher = darker.
    """
    if bg_kernel % 2 == 0:
        bg_kernel += 1
    bg = cv2.medianBlur(img8, bg_kernel).astype(np.float32)
    return (bg - img8.astype(np.float32)) / np.maximum(bg, 1.0)


def estimate_diameter(rel: np.ndarray, cut: float = 0.35) -> float:
    """Estimate the characteristic single-bead diameter (px) from an image.

    Uses the median area of compact, round objects so that clumps and debris
    do not distort the estimate. Falls back to a permissive area filter if
    too few clean objects are found.
    """
    mask = morphology.remove_small_objects(rel > cut, min_size=4)
    mask = ndi.binary_fill_holes(mask)
    lbl, n = ndi.label(mask)
    if n == 0:
        return 6.0  # safe default for a near-empty image
    props = measure.regionprops(lbl)
    areas = np.array([p.area for p in props])
    solidity = np.array([p.solidity for p in props])
    eccentricity = np.array([p.eccentricity for p in props])
    keep = (solidity > 0.88) & (eccentricity < 0.70) & (areas >= 4)
    if keep.sum() < 10:                 # not enough clean objects -> relax
        keep = areas >= 4
    median_area = float(np.median(areas[keep]))
    return 2.0 * np.sqrt(median_area / np.pi)


def count_image(
    img8: np.ndarray,
    diameter: Optional[float] = None,
    pix_cut: float = DEFAULTS["pix_cut"],
    peak_cut: float = DEFAULTS["peak_cut"],
    bg_kernel: int = DEFAULTS["bg_kernel"],
    log_snr: float = DEFAULTS["log_snr"],
) -> BeadResult:
    """Count beads in one grayscale image.

    Parameters
    ----------
    img8 : 2-D uint8 array
        Grayscale image, beads dark on a light background.
    diameter : float, optional
        Single-bead diameter in pixels. If None (default), estimated
        automatically with `estimate_diameter`. Provide a fixed value when
        analysing a new bead size to keep the scale constant across images.
    pix_cut, peak_cut, bg_kernel, log_snr :
        See DEFAULTS. The manuscript results use the defaults.

    Returns
    -------
    BeadResult
    """
    rel = relative_absorbance(img8, bg_kernel)
    if diameter is None:
        diameter = estimate_diameter(rel)

    bead_area = np.pi * (diameter / 2.0) ** 2
    sigma = (diameter / 2.0) / np.sqrt(2.0)          # LoG scale for this bead size
    min_sep = max(3, int(round(0.6 * diameter)))     # min distance between peaks
    min_size = max(4, int(round(0.25 * bead_area)))  # min object size in the mask

    # --- Candidate detection: Laplacian-of-Gaussian blob response ----------
    img = img8.astype(np.float32)
    log = ndi.gaussian_laplace(img, sigma) * (sigma ** 2)   # scale-normalised LoG
    med = np.median(log)
    mad = np.median(np.abs(log - med))
    snr = (log - med) / max(1.4826 * mad, 1e-6)             # robust SNR
    cand = feature.peak_local_max(
        snr, min_distance=min_sep, threshold_abs=log_snr, exclude_border=False
    )
    cand_abs = np.array([rel[y, x] for y, x in cand]) if len(cand) else np.array([])

    # --- Foreground mask ---------------------------------------------------
    bmask = morphology.remove_small_objects(rel > pix_cut, min_size=min_size)
    bmask = ndi.binary_fill_holes(bmask)

    # keep only candidates that are dark enough and inside the mask
    peaks = np.array(
        [p for p, a in zip(cand, cand_abs) if a > peak_cut and bmask[p[0], p[1]]]
    )

    lbl, n = ndi.label(bmask)
    props = measure.regionprops(lbl, intensity_image=rel)

    # peaks per labelled region
    npk = np.zeros(n + 1, int)
    for y, x in peaks:
        npk[lbl[y, x]] += 1

    # valid regions: at least one peak, and not an obvious linear artefact
    valid = [
        p for p in props
        if npk[p.label] >= 1
        and not (p.area > 4 * bead_area and p.solidity < 0.50 and p.eccentricity > 0.92)
    ]

    # isolated single beads used to calibrate single-bead area & mass
    iso = [
        p for p in valid
        if npk[p.label] == 1 and p.solidity > 0.88 and p.eccentricity < 0.70
    ]
    if len(iso) < 5:                      # relax if too few clean singles
        iso = [p for p in valid if npk[p.label] == 1]

    single_area = float(np.median([p.area for p in iso]))
    single_mass = float(np.median([p.intensity_image[p.image].sum() for p in iso]))

    total_area = float(sum(p.area for p in valid))
    total_mass = float(sum(p.intensity_image[p.image].sum() for p in valid))

    n_peak = int(len(peaks))
    n_area = total_area / single_area
    n_mass = total_mass / single_mass
    best = int(round((n_area + n_mass) / 2.0))

    return BeadResult(
        best=best, n_peak=n_peak, n_area=n_area, n_mass=n_mass,
        diameter=diameter, single_area=single_area,
        rel=rel, peaks=peaks, valid=valid, npk=npk, cand_abs=cand_abs, img=img8,
    )


def overlay_image(res: BeadResult) -> np.ndarray:
    """Draw detected beads on the original image.

    Red circle  = one resolved bead.
    Orange circle + number = a clump, labelled with its estimated bead count.
    """
    img8 = res.img
    ov = np.stack([img8] * 3, -1).astype(np.uint8)
    single_area = res.single_area
    for p in res.valid:
        cy, cx = map(int, p.centroid)
        if res.npk[p.label] == 1:
            cv2.circle(ov, (cx, cy), max(6, int(res.diameter)), (255, 40, 40), 2)
        else:
            est = max(res.npk[p.label], int(round(p.area / single_area)))
            r = max(9, int(np.sqrt(p.area / np.pi)) + 5)
            cv2.circle(ov, (cx, cy), r, (255, 170, 0), 2)
            cv2.putText(ov, str(est), (cx + r, cy - r),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 170, 0), 2)
    return ov


In [ ]:
# Upload your images. A file picker appears when this cell runs.
# (Outside Colab, images are read from an ./images folder instead.)

try:
    from google.colab import files
    print("Click 'Choose Files' and pick your bead image(s):")
    uploaded = files.upload()
    IN_COLAB = True
except ModuleNotFoundError:
    # Running outside Colab: read every image in an ./images folder instead.
    IN_COLAB = False
    uploaded = {}
    folder = "images"
    for fn in sorted(os.listdir(folder)) if os.path.isdir(folder) else []:
        if fn.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")):
            uploaded[fn] = open(os.path.join(folder, fn), "rb").read()
    print(f"Not in Colab — loaded {len(uploaded)} image(s) from the ./{folder} folder.")

print(f"Got {len(uploaded)} image(s):", ", ".join(uploaded.keys()) or "(none)")


In [ ]:
# Count the images, show results, and save counts.csv + results.zip.

os.makedirs("results", exist_ok=True)
rows = []
print(f"{'image':30}{'BEST':>7}{'peak':>7}{'area':>7}{'mass':>7}{'diam':>7}")
print("-" * 65)

for name, data in uploaded.items():
    img8 = np.array(Image.open(io.BytesIO(data)).convert("L")).astype(np.uint8)
    res = count_image(img8)
    stem = os.path.splitext(name)[0]
    print(f"{stem[:29]:30}{res.best:>7}{res.n_peak:>7}"
          f"{res.n_area:>7.0f}{res.n_mass:>7.0f}{res.diameter:>7.1f}")
    rows.append([stem, res.best, res.n_peak, round(res.n_area),
                 round(res.n_mass), round(res.diameter, 2)])

    ov = overlay_image(res)
    cv2.imwrite(f"results/{stem}_overlay.png", cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    ax[0].imshow(ov)
    ax[0].set_title(f"{stem}  -  ~{res.best} beads\n(red = single, orange = clump)")
    ax[0].axis("off")
    ax[1].hist(res.cand_abs, bins=np.arange(-0.05, 1.0, 0.025),
               color="#5a7fa6", edgecolor="white")
    ax[1].axvline(0.40, color="crimson", ls="--", lw=2, label="threshold (0.40)")
    ax[1].set_yscale("log")
    ax[1].set_xlabel("relative absorbance")
    ax[1].set_title("QC: dashed line should sit in the gap\nbetween the two humps")
    ax[1].legend()
    plt.savefig(f"results/{stem}_histogram.png", dpi=130, bbox_inches="tight")
    plt.show()

with open("results/counts.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image", "beads_best", "n_peak", "n_area", "n_mass", "bead_diameter_px"])
    w.writerows(rows)

print("\nBEST = the number to report.  peak = lower bound (resolvable beads only).")

if IN_COLAB:
    with zipfile.ZipFile("results.zip", "w") as z:
        for root, _, fnames in os.walk("results"):
            for fn in fnames:
                z.write(os.path.join(root, fn))
    from google.colab import files
    print("Downloading results.zip ...")
    files.download("results.zip")
else:
    print("Results saved in the ./results folder.")
